# Fase 3 — Estructura de la YouTube Analytics API (simulada)

**Objetivo:** documentar con precisión qué pediríamos a la Analytics API para Golazo (retención, audiencia, ingresos) y qué FORMA exacta tendría la respuesta, replicando el formato real (`youtubeAnalytics#resultTable`, `columnHeaders`, `rows`). Esto es el contrato que usaremos en la Fase 4 para rellenar `rows` con datos sintéticos coherentes.

**Requisito previo:** haber ejecutado, desde la raíz del proyecto:
```
python -m src.oauth_simulado
python -m src.analytics_client
```
Esto genera en `/raw`:
- `golazo_oauth_token_simulado.json`
- `golazo_analytics_contratos.json`

In [ ]:
import json
import os

RAW_DIR = os.path.join("..", "raw")

with open(os.path.join(RAW_DIR, "golazo_oauth_token_simulado.json"), encoding="utf-8") as f:
    token = json.load(f)

with open(os.path.join(RAW_DIR, "golazo_analytics_contratos.json"), encoding="utf-8") as f:
    contratos = json.load(f)

print("Informes definidos:", list(contratos.keys()))

## 1. Revisar el token OAuth simulado

Comprobamos que el token simulado tiene la forma correcta (scopes, canal autorizado, caducidad) antes de "usarlo" conceptualmente en las siguientes llamadas.

In [ ]:
print("Rol concedido:", token["canal_autorizado"]["rol_concedido"])
print("Canal autorizado:", token["canal_autorizado"]["channel_title"])
print("Scopes concedidos:")
for scope in token["scopes_concedidos"]:
    print(" -", scope)

## 2. Inspeccionar cada informe: petición + forma de la respuesta

Para cada informe (`evolucion_diaria`, `retencion_audiencia`, `demografia`, `trafico`, `ingresos`), mostramos los parámetros que se enviarían a `reports.query` y las columnas exactas (con su `columnType` y `dataType`) que devolvería la API.

In [ ]:
for nombre, info in contratos.items():
    print(f"\n=== {nombre.upper()} ===")
    print("Descripción:", info["descripcion"])
    print("Petición (params de reports.query):")
    for k, v in info["peticion"].items():
        print(f"    {k}: {v}")
    print("Columnas de la respuesta:")
    for col in info["forma_respuesta"]["columnHeaders"]:
        print(f"    {col['name']:35s} | {col['columnType']:9s} | {col['dataType']}")

## 3. Convertir la forma de un informe a DataFrame vacío

Como `rows` está vacío (aún no hay datos, ni reales ni sintéticos), construimos un `DataFrame` vacío pero **con las columnas y tipos correctos**, que es exactamente el esqueleto que rellenaremos en la Fase 4.

In [ ]:
import pandas as pd

TIPO_PANDAS = {"STRING": "object", "INTEGER": "int64", "FLOAT": "float64"}


def forma_a_dataframe_vacio(forma_respuesta: dict) -> pd.DataFrame:
    columnas = {
        col["name"]: pd.Series(dtype=TIPO_PANDAS[col["dataType"]])
        for col in forma_respuesta["columnHeaders"]
    }
    return pd.DataFrame(columnas)


df_evolucion_vacio = forma_a_dataframe_vacio(contratos["evolucion_diaria"]["forma_respuesta"])
print(df_evolucion_vacio.dtypes)
df_evolucion_vacio

## 4. Conclusiones para la Fase 4

1. Cada informe tiene su propio esqueleto de columnas y tipos, ya validado contra el formato real documentado por Google (`youtubeAnalytics#resultTable`).
2. En la Fase 4, generaremos datos sintéticos **fila a fila** respetando estos tipos (p. ej. `views` como entero, `averageViewDuration` como flotante, `day` como fecha en formato `STRING` `YYYY-MM-DD`).
3. El informe `evolucion_diaria` deberá cubrir el mismo rango de fechas que los vídeos reales extraídos de `@PuroBalompie` en la Fase 2, para poder cruzar ambas fuentes con coherencia temporal.
4. El informe `retencion_audiencia` requiere generar una curva por vídeo (no un único valor), ya que su dimensión es `elapsedVideoTimeRatio` (múltiples puntos entre 0 y 1 por vídeo).
5. `demografia` y `trafico` son distribuciones (deben sumar ~100% o el total de vistas del periodo, respectivamente), así que su generación sintética deberá normalizarse.